# 03 — Filtrado y Manipulación de Detecciones

## ¿Qué vamos a construir hoy?

Aprenderás a seleccionar exactamente las detecciones que necesitas:
por clase, por confianza, por tamaño, o por posición.

**Aprenderás a:**
- Filtrar sv.Detections con condiciones booleanas (como Excel)
- Eliminar detecciones duplicadas con NMS
- Combinar detecciones de múltiples fuentes

**Tiempo estimado:** 25 minutos

## ⏱️ Estructura de la Clase (Duración estimada: 1 hora)
- **Introducción y Conceptos Base**: 15 min
- **Desarrollo y Demostración Práctica**: 25 min
- **Análisis y Casos Extremos (Pausa y Observa)**: 20 min

## sv.Detections como tabla filtrable

Imagina una hoja de cálculo con una fila por objeto detectado:

| xyxy          | confidence | class_id |
|---------------|------------|----------|
| [10,20,50,80] | 0.92       | 0        |
| [100,30,200,120]| 0.71     | 0        |
| [300,10,600,400]| 0.85     | 5        |

Filtrar detecciones = seleccionar filas de esa tabla.
`detections[detections.class_id == 0]` = "dame solo las filas donde class_id es 0".

La sintaxis es idéntica a filtrar un array NumPy.

In [ ]:
!pip install supervision ultralytics
import supervision as sv
from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
import urllib.request
from pathlib import Path

Path("assets").mkdir(exist_ok=True)
urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "assets/bus.jpg")
image = cv2.imread("assets/bus.jpg")

model = YOLO("yolov8n.pt")
results = model(image)[0]
detections = sv.Detections.from_ultralytics(results)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

def mostrar(det, titulo, img=None):
    """Visualiza detecciones — helper para no repetir código de matplotlib en cada celda."""
    if img is None:
        img = image
    etiquetas = [f"{results.names[c]}" for c in det.class_id]
    scene = box_annotator.annotate(scene=img.copy(), detections=det)
    scene = label_annotator.annotate(scene=scene, detections=det, labels=etiquetas)
    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(f"{titulo}  ({len(det)} objetos)")
    plt.show()

mostrar(detections, "Todas las detecciones (punto de partida)")

## Filtrado por confianza

In [ ]:
# La comparación crea una máscara booleana (array de True/False) para cada detección
# sv.Detections[mascara_booleana] devuelve solo las filas donde la máscara es True
mascara = detections.confidence > 0.5
alta_confianza = detections[mascara]

print(f"Total: {len(detections)} | Con confianza > 0.5: {len(alta_confianza)}")
mostrar(alta_confianza, "Solo confianza > 0.5")

## Filtrado por clase

In [ ]:
# Clases disponibles en esta imagen (COCO dataset)
print("Objetos detectados:")
for class_id in sorted(set(detections.class_id)):
    n = (detections.class_id == class_id).sum()
    print(f"  Clase {class_id} ({results.names[class_id]}): {n} detecciones")

# Clase 0 = 'person' en COCO
personas = detections[detections.class_id == 0]
print(f"\nSolo personas: {len(personas)}")
mostrar(personas, "Solo personas (clase 0)")

In [ ]:
# Para combinar condiciones, DEBES usar & (AND elemento a elemento), no 'and'
# 'and' en Python es para valores booleanos individuales, no arrays
personas_seguras = detections[
    (detections.class_id == 0) & (detections.confidence > 0.6)
]
print(f"Personas con confianza > 60%: {len(personas_seguras)}")
mostrar(personas_seguras, "Personas con confianza > 60%")

## NMS — Non-Maximum Suppression

A veces el modelo detecta el mismo objeto múltiples veces con cajas ligeramente diferentes.
NMS conserva solo la caja de mayor confianza cuando hay demasiado solapamiento.

In [ ]:
# Generamos duplicados artificiales: mismo modelo, dos umbrales de confianza diferentes
# conf=0.3 detecta más objetos (incluyendo muchos con baja confianza)
# conf=0.7 detecta menos pero con mayor certeza
# Al mergear ambos, obtenemos el mismo objeto detectado dos veces
results_baja = model(image, conf=0.3)[0]
results_alta = model(image, conf=0.7)[0]
det_baja = sv.Detections.from_ultralytics(results_baja)
det_alta = sv.Detections.from_ultralytics(results_alta)

mezclado = sv.Detections.merge([det_baja, det_alta])
print(f"Detecciones individuales: baja_conf={len(det_baja)}, alta_conf={len(det_alta)}")
print(f"Después de merge (duplicados incluidos): {len(mezclado)}")

# threshold=0.5 → si dos cajas se solapan más del 50%, NMS elimina la de menor confianza
sin_duplicados = mezclado.with_nms(threshold=0.5)
print(f"Después de NMS (sin duplicados): {len(sin_duplicados)}")

mostrar(mezclado,       "Antes de NMS (con duplicados)")
mostrar(sin_duplicados, "Después de NMS (threshold=0.5)")

## Filtrado por tamaño

In [ ]:
# detections.area calcula automáticamente el área de cada bounding box en píxeles²
areas = detections.area
print(f"Área mínima:  {areas.min():.0f} px²")
print(f"Área máxima:  {areas.max():.0f} px²")
print(f"Área promedio: {areas.mean():.0f} px²")

# Objetos muy pequeños pueden ser falsos positivos o ruido
objetos_grandes = detections[detections.area > 5000]
print(f"\nObjetos con área > 5000 px²: {len(objetos_grandes)}")
mostrar(objetos_grandes, "Solo objetos grandes (área > 5000 px²)")

## 🔧 Exploración interactiva

### Experimento 1: Merge + NMS con diferentes umbrales

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, thresh in zip(axes, [0.3, 0.5, 0.8]):
    filtered = sv.Detections.merge([det_baja, det_alta]).with_nms(threshold=thresh)
    etiquetas = [results.names[c] for c in filtered.class_id]
    scene = box_annotator.annotate(scene=image.copy(), detections=filtered)
    scene = label_annotator.annotate(scene=scene, detections=filtered, labels=etiquetas)
    ax.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
    ax.set_title(f"NMS threshold={thresh}\n({len(filtered)} objetos)")
    ax.axis("off")
plt.tight_layout()
plt.show()
# 💭 Reflexión: threshold más bajo → más estricto (menos objetos).
# ¿Por qué? Porque permite menos solapamiento antes de eliminar una caja.

### Experimento 2: Excluir una clase

In [ ]:
# Clase 5 = 'bus' en COCO
# != excluye en lugar de incluir — la lógica es idéntica a la inclusión
sin_buses = detections[detections.class_id != 5]
print(f"Con buses: {len(detections)} | Sin buses: {len(sin_buses)}")
mostrar(sin_buses, "Sin autobuses (clase 5 excluida)")
# 💭 Reflexión: ¿Qué otras clases podrías excluir o filtrar?
# Prueba cambiar el 5 por otro class_id de la tabla de arriba.

### Experimento 3: Las N detecciones más confiables

In [ ]:
# np.argsort devuelve los índices que ordenarían el array de menor a mayor
# [::-1] invierte el orden (de mayor a menor)
# [:3] toma los primeros 3 índices
indices_top3 = np.argsort(detections.confidence)[::-1][:3]
top3 = detections[indices_top3]

print("Top 3 detecciones por confianza:")
for i in range(len(top3)):
    print(f"  {results.names[top3.class_id[i]]}: {top3.confidence[i]:.1%}")

mostrar(top3, "Top 3 detecciones más confiables")
# 💭 Reflexión: ¿Son siempre los objetos más grandes los más confiables?
# No necesariamente — depende del entrenamiento del modelo y la imagen.

## 🚀 Reto de extensión

**Tarea:** Filtra las detecciones para quedarte solo con los objetos que están
en la **mitad derecha** de la imagen (su centro x > ancho_imagen / 2).

**Pista:** El centro x de cada caja es:
```python
centros_x = (detections.xyxy[:, 0] + detections.xyxy[:, 2]) / 2
# Ahora crea la máscara y filtra
```

In [ ]:
# Escribe tu solución aquí
centros_x = (detections.xyxy[:, 0] + detections.xyxy[:, 2]) / 2
mitad_imagen = image.shape[1] / 2
mascara = centros_x > mitad_imagen
# detecciones_derecha = ...